# Ask Aziz: retrieval evaluation

This notebook re-scores the **retriever** behind the Ask Aziz question box.

- **What it checks:** for each judged question, did the retriever put a passage a good answer must show at rank 1, or in the top 3? And did it correctly return nothing when there is no evidence?
- **What it does not do:** it does not run the retriever (that is TypeScript, `src/retrieval/`), and there is no language model anywhere. Ask Aziz retrieves passages; it does not generate answers, so this is the *retrieval* half of a RAG system only.
- **Inputs:** `content/judged.yaml` (questions and the passages each should find) and `reports/retrieval-keyword.json` (what the retriever returned, written by `npm run eval:retrieval`).
- **AWS:** no AWS service is used here. This runs locally.

To refresh the saved run first: `npm run eval:retrieval`, then re-run this notebook.

In [1]:
import json
from pathlib import Path
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
judged = yaml.safe_load((ROOT / "content" / "judged.yaml").read_text(encoding="utf-8"))
saved = json.loads((ROOT / "reports" / "retrieval-keyword.json").read_text(encoding="utf-8"))
runs = {row["id"]: row for row in saved["rows"]}
corpus = json.loads((ROOT / "src" / "generated" / "corpus.json").read_text(encoding="utf-8"))
passages = corpus["passages"] if isinstance(corpus, dict) and "passages" in corpus else corpus

print(f"{len(judged)} judged questions, {len(passages)} passages in the corpus")
missing = [q["id"] for q in judged if q["id"] not in runs]
assert not missing, f"judged questions with no saved run (re-run npm run eval:retrieval): {missing}"


30 judged questions, 23 passages in the corpus


## Recompute the metrics from scratch

Definitions (same as the project's script):

- **rank** = position of the first relevant passage in the returned list (1 = best).
- **hit@1 / hit@3** = share of answerable questions whose rank is 1 / at most 3.
- **MRR** = mean of 1/rank over answerable questions.
- **Abstention is correct** when the question has no relevant passage *and* the retriever returned none.

In [2]:
rows = []
for q in judged:
    top = runs[q["id"]]["top"]
    relevant = set(q["relevant"])
    rank = next((i + 1 for i, pid in enumerate(top) if pid in relevant), None)
    rows.append({"id": q["id"], "question": q["question"], "relevant": q["relevant"],
                 "top": top, "rank": rank, "answerable": bool(relevant)})

answerable = [r for r in rows if r["answerable"]]
abstain = [r for r in rows if not r["answerable"]]

hit1 = sum(r["rank"] == 1 for r in answerable) / len(answerable)
hit3 = sum(r["rank"] is not None and r["rank"] <= 3 for r in answerable) / len(answerable)
mrr = sum(1 / r["rank"] for r in answerable if r["rank"]) / len(answerable)
abstain_ok = sum(not r["top"] for r in abstain)

print(f"answerable questions : {len(answerable)}")
print(f"hit@1                : {hit1:.3f}")
print(f"hit@3                : {hit3:.3f}")
print(f"MRR                  : {mrr:.3f}")
print(f"correct abstentions  : {abstain_ok}/{len(abstain)}")


answerable questions : 27
hit@1                : 0.963
hit@3                : 1.000
MRR                  : 0.981
correct abstentions  : 3/3


## Cross-check against the numbers the project's own script saved

In [3]:
checks = {
    "answerable": (len(answerable), saved["answerable"]),
    "hit1": (hit1, saved["hit1"]),
    "hit3": (hit3, saved["hit3"]),
    "mrr": (mrr, saved["mrr"]),
    "abstainOk": (abstain_ok, saved["abstainOk"]),
    "abstainTotal": (len(abstain), saved["abstainTotal"]),
}
for name, (mine, theirs) in checks.items():
    ok = abs(mine - theirs) < 1e-9
    print(f"{'OK ' if ok else 'DIFF'} {name:13} notebook={mine:.6g}  saved={theirs:.6g}")
assert all(abs(a - b) < 1e-9 for a, b in checks.values()), "notebook and saved report disagree"


OK  answerable    notebook=27  saved=27
OK  hit1          notebook=0.962963  saved=0.962963
OK  hit3          notebook=1  saved=1
OK  mrr           notebook=0.981481  saved=0.981481
OK  abstainOk     notebook=3  saved=3
OK  abstainTotal  notebook=3  saved=3


## Every question

In [4]:
print(f"{'id':4} {'rank':>4}  {'question':58} top passages")
for r in rows:
    rank = "-" if r["rank"] is None else r["rank"]
    flag = "" if (r["rank"] == 1 or (not r["answerable"] and not r["top"])) else "  <-- not rank 1"
    print(f"{r['id']:4} {str(rank):>4}  {r['question'][:56]:58} {', '.join(r['top']) or '(none)'}{flag}")


id   rank  question                                                   top passages
q01     1  Is TraffiX deployed in Tunis?                              tr-simulation
q02     1  How do you keep pedestrians safe at crossings?             tr-pedestrian
q03     1  Are ambulances faster with TraffiX?                        tr-ambulance
q04     1  What did you personally build in the TraffiX team?         tr-role
q05     1  Are the traffic results statistically significant?         tr-mixed, fr-p50
q06     1  How do you decide which pages editors should review firs   fr-question
q07     2  What precision did the ranking model reach?                fr-question, fr-p50, portfolio-weakest  <-- not rank 1
q08     1  How did you avoid data leakage?                            fr-leakage
q09     1  Does refreshing a page cause more traffic?                 fr-question, fr-p50
q10     1  How do you process scanned documents?                      hr-pipeline, hr-accuracy, pr-experience
q11     1  Wh

## Where it is not perfect

In [5]:
weak = [r for r in answerable if r["rank"] != 1]
wrong_abstain = [r for r in abstain if r["top"]]
if not weak and not wrong_abstain:
    print("No misses on this set. See the limitations below before reading that as good news.")
for r in weak:
    print(f"{r['id']}: {r['question']}\n   wanted {r['relevant']}, got {r['top']} (first relevant at rank {r['rank']})")
for r in wrong_abstain:
    print(f"{r['id']}: should have returned nothing but returned {r['top']}")


q07: What precision did the ranking model reach?
   wanted ['fr-p50'], got ['fr-question', 'fr-p50', 'portfolio-weakest'] (first relevant at rank 2)


## Limitations (read these before quoting the scores)

- **I wrote both sides.** I wrote the judged questions *and* the tags on the passages. A retriever tested on questions its author wrote will look better than it would with other people's questions. Nobody else has tested it.
- **Small set.** A few dozen questions over a corpus of about two dozen passages (the first cell prints the exact counts). One question moving rank changes hit@1 by several points.
- **Typo questions are my own typos.** The misspelled questions are ones I invented; real visitors will type differently.
- **Retrieval only.** The score says nothing about answer quality, because no answers are generated here.
- **This notebook re-scores a saved run.** It proves the arithmetic is right and matches the project script. It does not re-execute the retriever; `npm run eval:retrieval` does that.